# Лабораторная работа №4
## Выполнил студент группы БФИ2201 Балыко Владислав Васильевич

### Цель

Реализовать методы поиска в соответствии с заданием. Организовать генерацию начального набора случайных данных. Для всех вариантов добавить реализацию добавления, поиска и удаления элементов. Оценить время работы каждого алгоритма поиска и сравнить его со временем работы стандартной функции поиска, используемой в выбранном языке программирования.

### Оглавление
1. Алгоритмы поиска
2. Хеш-таблицы и методы разрешения коллизий
3. Задача о восьми ферзях
4. Сравнение времени работы
5. Вывод


### Алгоритмы:

Бинарный поиск | Бинарное дерево | Фибоначчиев | Интерполяционный| Простое рехэширование | Рехэширование с помощью псевдослучайных чисел | Метод цепочек



### Задание 1

In [1]:
import random
import timeit

In [2]:
# Сгенерировать массив
def generate_arr():
    length = int(input("Введите длину массива: "))
    min_val = int(input("Введите минимальное число в массиве: "))
    max_val = int(input("Введите максимальное число в массиве: "))
    arr = [random.randint(min_val, max_val) for _ in range(length)]
    arr.sort()
    print("Сгенерированный массив:", arr)
    return arr


# Генерация массива
print("Исходный массив\n__________________________")
array = generate_arr()
find_elem = int(input("Введите элемент, который хотите найти: "))
print(f'Элемент который нужно найти: {find_elem}')

Исходный массив
__________________________
Сгенерированный массив: [2, 4, 9, 10, 11, 12, 12, 13, 14, 15, 15, 15, 15, 16, 17, 19, 20, 21, 23, 24, 26, 26, 26, 26, 27, 27, 28, 29, 30, 30, 32, 32, 34, 35, 35, 36, 36, 36, 37, 38, 38, 39, 41, 41, 41, 42, 43, 44, 44, 45, 45, 46, 46, 46, 47, 47, 47, 50, 50, 50, 50, 51, 52, 52, 54, 55, 56, 56, 57, 57, 58, 59, 61, 62, 62, 63, 64, 65, 66, 67, 67, 71, 71, 71, 73, 76, 77, 77, 79, 79, 80, 80, 80, 81, 81, 82, 84, 86, 88, 88]
Элемент который нужно найти: 20


In [3]:
# Бинарный поиск
def binary_search(arr, x):
    l, r = 0, len(arr) - 1
    while l <= r:
        m = (l + r) // 2
        if arr[m] < x:
            l = m + 1
        elif arr[m] > x:
            r = m - 1
        else:
            return m
    return -1

In [4]:
# Класс узла
class Node:
    def __init__(self, key):
        self.left = None
        self.right = None
        self.val = key


# Класс бинарного дерева
class BinaryTree:
    def __init__(self):
        self.root = None

    def insert(self, key):
        if self.root is None:
            self.root = Node(key)
        else:
            self._insert(key, self.root)

    def _insert(self, key, node):
        if key < node.val:
            if node.left is None:
                node.left = Node(key)
            else:
                self._insert(key, node.left)
        else:
            if node.right is None:
                node.right = Node(key)
            else:
                self._insert(key, node.right)

    def search(self, key):
        return self._search(key, self.root)

    def _search(self, key, node):
        if node is None or node.val == key:
            return node
        elif key < node.val:
            return self._search(key, node.left)
        else:
            return self._search(key, node.right)

    def _find_min(self, node):
        current = node
        while current.left is not None:
            current = current.left
        return current

    def delete(self, key):
        self.root = self._delete(key, self.root)

    def _delete(self, key, node):
        if node is None:
            return node
        if key < node.val:
            node.left = self._delete(key, node.left)
        elif key > node.val:
            node.right = self._delete(key, node.right)
        else:
            if node.left is None:
                return node.right
            elif node.right is None:
                return node.left
            temp = self._find_min(node.right)
            node.val = temp.val
            node.right = self._delete(temp.val, node.right)
        return node

    def print_tree(self):
        result = []
        self._print_tree(self.root, result)
        return result

    def _print_tree(self, node, result):
        if node:
            self._print_tree(node.left, result)
            result.append(node.val)
            self._print_tree(node.right, result)


# Тестирование бинарного дерева
bst = BinaryTree()
for num in [12, 32, 78, 23, 14, 56]:
    bst.insert(num)
print("Бинарное дерево (in-order):", bst.print_tree())

Бинарное дерево (in-order): [12, 14, 23, 32, 56, 78]


In [5]:
# Фибоначчиев поиск
def fibonacci_search(arr, x):
    fib2, fib1 = 0, 1
    fib = fib1 + fib2
    while fib < len(arr):
        fib2, fib1 = fib1, fib
        fib = fib1 + fib2
    offset = -1
    while fib > 1:
        i = min(offset + fib2, len(arr) - 1)
        if arr[i] < x:
            fib, fib1 = fib1, fib2
            fib2 = fib - fib1
            offset = i
        elif arr[i] > x:
            fib, fib1 = fib2, fib1 - fib2
            fib2 = fib - fib1
        else:
            return i
    if fib1 and arr[offset + 1] == x:
        return offset + 1
    return -1

In [6]:
# Интерполяционный поиск
def interpolation_search(arr, x):
    low, high = 0, len(arr) - 1
    while low <= high and arr[low] <= x <= arr[high]:
        if low == high:
            if arr[low] == x:
                return low
            return -1
        if arr[high] == arr[low]:
            return low if arr[low] == x else -1
        pos = low + ((x - arr[low]) * (high - low) // (arr[high] - arr[low]))
        if arr[pos] == x:
            return pos
        if arr[pos] < x:
            low = pos + 1
        else:
            high = pos - 1
    return -1


In [7]:
# Измерение времени работы алгоритмов
starttime = timeit.default_timer()
binary_search(array, find_elem)
endtime = timeit.default_timer()
print("Бинарный поиск:", endtime - starttime, "секунд")

starttime = timeit.default_timer()
bst.search(find_elem)
endtime = timeit.default_timer()
print("Бинарное дерево:", endtime - starttime, "секунд")

starttime = timeit.default_timer()
fibonacci_search(array, find_elem)
endtime = timeit.default_timer()
print("Фибоначчиев поиск:", endtime - starttime, "секунд")

starttime = timeit.default_timer()
interpolation_search(array, find_elem)
endtime = timeit.default_timer()
print("Интерполяционный поиск:", endtime - starttime, "секунд")

starttime = timeit.default_timer()
array.index(find_elem)
endtime = timeit.default_timer()
print("Стандартный поиск:", endtime - starttime, "секунд")

Бинарный поиск: 9.797399980016053e-05 секунд
Бинарное дерево: 0.00011893400005646981 секунд
Фибоначчиев поиск: 9.937700087903067e-05 секунд
Интерполяционный поиск: 9.170199882646557e-05 секунд
Стандартный поиск: 7.189600000856444e-05 секунд


### Задание 2

In [8]:
# Простое рехэширование
class HashTableP:
    DELETED = object()

    def __init__(self):
        self.size = 10
        self.keys = [None] * self.size
        self.values = [None] * self.size

    def hash_function(self, key):
        return key % self.size

    def put(self, key, data):
        index = self.hash_function(key)
        first_deleted = None
        while self.keys[index] is not None:
            if self.keys[index] == key:
                self.values[index] = data
                return
            if self.keys[index] is self.DELETED and first_deleted is None:
                first_deleted = index
            index = (index + 1) % self.size
        target = first_deleted if first_deleted is not None else index
        self.keys[target] = key
        self.values[target] = data

    def get(self, key):
        index = self.hash_function(key)
        start = index
        while self.keys[index] is not None:
            if self.keys[index] != self.DELETED and self.keys[index] == key:
                return self.values[index]
            index = (index + 1) % self.size
            if index == start:
                break
        return None

    def delete(self, key):
        index = self.hash_function(key)
        start = index
        while self.keys[index] is not None:
            if self.keys[index] != self.DELETED and self.keys[index] == key:
                self.keys[index] = self.DELETED
                self.values[index] = None
                return
            index = (index + 1) % self.size
            if index == start:
                break


In [9]:
# Рехэширование с псевдослучайными числами
class HashTableR:
    DELETED = object()

    def __init__(self):
        self.size = 10
        self.keys = [None] * self.size
        self.values = [None] * self.size

    def step_function(self, key):
        return 1 + (key % (self.size - 1))

    def put(self, key, data):
        index = key % self.size
        step = self.step_function(key)
        first_deleted = None
        start = index
        while self.keys[index] is not None:
            if self.keys[index] == key:
                self.values[index] = data
                return
            if self.keys[index] is self.DELETED and first_deleted is None:
                first_deleted = index
            index = (index + step) % self.size
            if index == start:
                break
        target = first_deleted if first_deleted is not None else index
        self.keys[target] = key
        self.values[target] = data

    def get(self, key):
        index = key % self.size
        step = self.step_function(key)
        start = index
        while self.keys[index] is not None:
            if self.keys[index] != self.DELETED and self.keys[index] == key:
                return self.values[index]
            index = (index + step) % self.size
            if index == start:
                break
        return None

    def delete(self, key):
        index = key % self.size
        step = self.step_function(key)
        start = index
        while self.keys[index] is not None:
            if self.keys[index] != self.DELETED and self.keys[index] == key:
                self.keys[index] = self.DELETED
                self.values[index] = None
                return
            index = (index + step) % self.size
            if index == start:
                break


In [10]:
# Метод цепочек
class HashTableChain:
    class Node:
        def __init__(self, key, value):
            self.key = key
            self.value = value
            self.next = None

    def __init__(self):
        self.capacity = 10
        self.buckets = [None] * self.capacity

    def put(self, key, value):
        index = hash(key) % self.capacity
        new_node = self.Node(key, value)
        if not self.buckets[index]:
            self.buckets[index] = new_node
        else:
            current = self.buckets[index]
            while current.next:
                if current.key == key:
                    current.value = value
                    return
                current = current.next
            current.next = new_node

    def get(self, key):
        index = hash(key) % self.capacity
        current = self.buckets[index]
        while current:
            if current.key == key:
                return current.value
            current = current.next
        return None

    def delete(self, key):
        index = hash(key) % self.capacity
        current = self.buckets[index]
        prev = None
        while current:
            if current.key == key:
                if prev:
                    prev.next = current.next
                else:
                    self.buckets[index] = current.next
                return
            prev = current
            current = current.next

In [11]:
# Хеш-таблицы
print("Простое рехэширование:")
table_p = HashTableP()
for key, value in [(1, "a"), (2, "b"), (11, "c"), (21, "d")]:
    table_p.put(key, value)
print(table_p.get(1))
print(table_p.get(2))
print(table_p.get(11))
print(table_p.get(21))

print("\nМетод цепочек:")
table_chain = HashTableChain()
for key, value in [(1, "a"), (2, "b"), (11, "c"), (21, "d")]:
    table_chain.put(key, value)
print(table_chain.get(1))
print(table_chain.get(2))
print(table_chain.get(11))
print(table_chain.get(21))

print("\nРехэширование с псевдослучайными числами:")
table_r = HashTableR()
for key, value in [(1, "a"), (2, "b"), (11, "c"), (21, "d")]:
    table_r.put(key, value)
print(table_r.get(1))
print(table_r.get(2))
print(table_r.get(11))
print(table_r.get(21))

Простое рехэширование:
a
b
c
d

Метод цепочек:
a
b
c
d

Рехэширование с псевдослучайными числами:
a
b
c
d


### Задание 3

Расставить на стандартной 64-клеточной шахматной доске 8 ферзей так, чтобы ни один из них не находился под боем другого». Подразумевается, что ферзь бьёт все клетки, расположенные по вертикалям, горизонталям и обеим диагоналям.

Написать программу,  которая находит хотя бы один способ решения задач.

In [12]:
# Задача о восьми ферзях
def chess():
    board = [['-' for _ in range(8)] for _ in range(8)]
    cols = set()
    pos_diagonals = set()
    neg_diagonals = set()

    def place_queen(row):
        if row == 8:
            return True
        for col in range(8):
            if col not in cols and (row + col) not in pos_diagonals and (row - col) not in neg_diagonals:
                board[row][col] = 'Q'
                cols.add(col)
                pos_diagonals.add(row + col)
                neg_diagonals.add(row - col)
                if place_queen(row + 1):
                    return True
                board[row][col] = '-'
                cols.remove(col)
                pos_diagonals.remove(row + col)
                neg_diagonals.remove(row - col)
        return False

    place_queen(0)
    for row in board:
        print(' '.join(row))


chess()

Q - - - - - - -
- - - - Q - - -
- - - - - - - Q
- - - - - Q - -
- - Q - - - - -
- - - - - - Q -
- Q - - - - - -
- - - Q - - - -


### Вывод


В лабораторной работе были реализованы и сравнены алгоритмы поиска, структуры хеш-таблиц и решение задачи о восьми ферзях. Результаты показывают, что выбор алгоритма и структуры данных существенно влияет на производительность и корректность обработки данных.